# ✈️ AirTrack Beauvais – Notebook de méthode

Une journée type de trafic aérien à **Paris-Beauvais (LFOB)**, piste 12 : trajectoires 3D, trafic par heure, compagnies et appareils.

> ⚠️ **Jeu de données fictif.** Les trajectoires sont synthétiques (corridors tirés de la carte DGAC du 20 juin 2025, appliqués au planning réel du 12/10/2026). Le notebook démontre la **méthode** : avec de vraies données ADS-B au même format, rien ne change.

**Ce notebook fait, dans l'ordre :**
1. charge les deux fichiers CSV (trajectoires + vols) ;
2. contrôle leur qualité (zone de 25 NM, plafond, continuité) ;
3. calcule le **trafic par heure** et le nombre d'avions simultanés dans la zone ;
4. résume **compagnies, types d'appareils, destinations** ;
5. trace les **profils d'altitude et de vitesse** en approche / décollage ;
6. produit une **carte 3D animée** (curseur temporel) ;
7. exporte des tables prêtes à l'emploi pour l'application Streamlit.

**Fichiers à fournir** (à glisser dans Colab quand on vous le demande) :
`trajectoires_LFOB_2026-10-12.csv` et `vols_LFOB_2026-10-12.csv`.

## 1. Préparation

In [ ]:
import importlib, subprocess, sys
for pkg in ["pandas", "numpy", "matplotlib", "plotly", "scipy"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Environnement prêt")

## 2. Chargement des données

In [ ]:
import os
import numpy as np
import pandas as pd

DATE = "2026-10-12"
F_TRAJ = f"trajectoires_LFOB_{DATE}.csv"
F_VOLS = f"vols_LFOB_{DATE}.csv"

# Repère géographique : centre de la piste 12/30 de Beauvais-Tillé
ARP_LAT, ARP_LON = 49.45225, 2.117795
KM_PER_DEG_LAT, KM_PER_DEG_LON = 111.22, 72.48     # à 49,45° N
KM_PER_NM = 1.852
ZONE_NM, CEILING_FT = 25, 15000
RWY_HDG, RWY_HALF_KM = 123.65, 1.214              # piste 12 (cap vrai)

try:                                               # Colab : proposer l'import des fichiers
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

missing = [f for f in (F_TRAJ, F_VOLS) if not os.path.exists(f)]
if missing:
    if IN_COLAB:
        print("Sélectionnez ces fichiers :", missing)
        files.upload()
    else:
        raise FileNotFoundError(f"Fichiers manquants : {missing}")

traj = pd.read_csv(F_TRAJ)
vols = pd.read_csv(F_VOLS)

# Position en km autour de l'aéroport (x = Est, y = Nord)
traj["x_km"] = (traj.lon - ARP_LON) * KM_PER_DEG_LON
traj["y_km"] = (traj.lat - ARP_LAT) * KM_PER_DEG_LAT

# On rattache les caractéristiques du vol à chaque point
cols = ["flight_id", "vol", "sens", "compagnie", "type_avion", "immat", "aeroport_lie_ville", "corridor"]
traj = traj.merge(vols[cols], on="flight_id", how="left").sort_values(["flight_id", "t_local_s"]).reset_index(drop=True)

print(f"{len(vols)} vols – {len(traj):,} points de trajectoire")
traj.head()

## 3. Contrôles qualité
Avant toute analyse, on vérifie que les données respectent le cahier des charges : zone de 25 NM, plafond, temps croissant, pas de « téléportation », vitesses et altitudes continues.

In [ ]:
g_ = traj.groupby("flight_id")
dt_ = g_.t_local_s.diff()
step = np.hypot(g_.x_km.diff(), g_.y_km.diff()) / dt_ * 3600 / KM_PER_NM   # vitesse « géométrique » (kt)
accel = (g_.gs_kt.diff() * 0.5144 / dt_).abs()

checks = {
    "Aucune valeur manquante": traj[["lat", "lon", "alt_ft", "gs_kt", "t_local_s"]].isna().sum().sum() == 0,
    "Tous les points dans le cercle de 25 NM": traj.dist_arp_nm.max() <= ZONE_NM + 0.1,
    f"Altitude max ≤ {CEILING_FT:,} ft".replace(",", " "): traj.alt_ft.max() <= CEILING_FT,
    "Temps strictement croissant pour chaque vol":
        traj.groupby("flight_id").t_local_s.apply(lambda s: s.is_monotonic_increasing).all(),
    "Vitesse sol cohérente avec le déplacement (écart moyen < 10 kt, en l'air)":
        (step - traj.gs_kt).abs()[(~traj.on_ground) & step.notna()].mean() < 10,
    "Accélération max < 4 m/s² (pas de saut de vitesse)": accel.max() < 4.0,
    "57 arrivées + 57 départs": (vols.sens.value_counts().to_dict() == {"ARR": 57, "DEP": 57}),
}
for name, ok in checks.items():
    print("✅" if ok else "❌", name)
assert all(checks.values()), "Un contrôle a échoué – inspecter les données avant d'aller plus loin"

print(f"\nDistance max à l'aéroport : {traj.dist_arp_nm.max():.2f} NM | altitude max : {traj.alt_ft.max():,} ft | "
      f"vitesse sol max : {traj.gs_kt.max()} kt")

## 4. Trafic par heure
Chaque vol compte à l'heure de son **événement piste** : atterrissage pour une arrivée, décollage pour un départ (heure locale, CEST).

In [ ]:
import plotly.graph_objects as go

vols["heure"] = vols.heure_evenement_retenue.str[:2].astype(int)
par_heure = (vols.pivot_table(index="heure", columns="sens", values="flight_id", aggfunc="count", fill_value=0)
             .reindex(range(24), fill_value=0).rename(columns={"ARR": "Arrivées", "DEP": "Départs"}))
par_heure["Total"] = par_heure["Arrivées"] + par_heure["Départs"]

top3 = par_heure.Total.nlargest(3)
print("Heures les plus chargées :", ", ".join(f"{h:02d}h ({n} mouvements)" for h, n in top3.items()))
print(f"Total : {par_heure.Total.sum()} mouvements | moyenne {par_heure.Total.mean():.1f}/h | "
      f"3 heures de pointe = {top3.sum() / par_heure.Total.sum():.0%} du trafic")
creuses = par_heure.index[par_heure.Total == 0].tolist()
print("Heures sans aucun mouvement :", ", ".join(f"{h:02d}h" for h in creuses) if creuses else "aucune")

fig = go.Figure()
fig.add_bar(x=par_heure.index, y=par_heure["Arrivées"], name="Arrivées", marker_color="#1f77b4")
fig.add_bar(x=par_heure.index, y=par_heure["Départs"], name="Départs", marker_color="#ff7f0e")
fig.update_layout(barmode="stack", title="Mouvements par heure locale – LFOB, 12/10/2026",
                  xaxis=dict(title="Heure locale", dtick=1), yaxis_title="Mouvements", height=420)
fig.show()
par_heure

**Avions présents simultanément dans la zone** (minute par minute) : un autre indicateur de charge, utile pour la lisibilité de la carte.

In [ ]:
presence = traj.groupby("flight_id").t_local_s.agg(["min", "max"])
minutes = np.arange(0, 24 * 60)
en_zone = np.array([((presence["min"] <= m * 60 + 30) & (presence["max"] >= m * 60 + 30)).sum() for m in minutes])
peak = en_zone.max()
peak_times = [f"{m // 60:02d}:{m % 60:02d}" for m in minutes[en_zone == peak][:5]]
print(f"Maximum : {peak} avions simultanément dans la zone (ex. à {', '.join(peak_times)}) | "
      f"moyenne sur la journée : {en_zone.mean():.2f}")

fig = go.Figure(go.Scatter(x=[f"{m // 60:02d}:{m % 60:02d}" for m in minutes], y=en_zone, mode="lines",
                           line=dict(color="#2ca02c", width=1.5), fill="tozeroy"))
fig.update_layout(title="Avions simultanément dans la zone de 25 NM", xaxis=dict(title="Heure locale", dtick=60),
                  yaxis_title="Avions", height=320)
fig.show()

## 5. Compagnies, appareils, destinations
Les proportions du jeu de données suivent le planning réel. Pour comparaison, les moyennes quotidiennes relevées sur Flightradar24 (capture fournie) : Ryanair ≈ 87, Wizz Air Malta ≈ 11, Wizz Air ≈ 7 vols/jour.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))

c = vols.compagnie.value_counts()
ax[0].barh(c.index[::-1], c.values[::-1], color="#00774a")
for i, v in enumerate(c.values[::-1]):
    ax[0].text(v + 0.8, i, f"{v}  ({v / len(vols):.0%})", va="center")
ax[0].set_title("Compagnies (mouvements)"); ax[0].set_xlim(0, c.max() * 1.25)

a = vols.type_avion.value_counts()
ax[1].bar(a.index, a.values, color="#1f4e79")
for i, v in enumerate(a.values):
    ax[1].text(i, v + 0.8, f"{v}\n({v / len(vols):.0%})", ha="center")
ax[1].set_title("Types d'appareils"); ax[1].set_ylim(0, a.max() * 1.2)

d = vols.aeroport_lie_ville.value_counts().head(10)
ax[2].barh(d.index[::-1], d.values[::-1], color="#b5651d")
ax[2].set_title("Top 10 liaisons (mouvements)")
plt.tight_layout(); plt.show()

print("Nombre d'avions (immatriculations) distincts :", vols.immat.nunique())
croise = pd.crosstab(vols.compagnie, vols.type_avion, margins=True, margins_name="Total")
croise

## 6. Profils d'altitude et de vitesse
Altitude et vitesse sol en fonction de la distance à l'aéroport : médiane et enveloppe 10-90 % pour les **arrivées** (descente, ralentissement, finale ILS) et les **départs** (accélération, montée).

In [ ]:
import matplotlib.pyplot as plt

air = traj[~traj.on_ground].copy()
air["bin"] = air.dist_arp_nm.round().astype(int)
fig, ax = plt.subplots(2, 2, figsize=(14, 8), sharex="col")
for j, (sens, titre, col) in enumerate([("ARR", "Arrivées", "#1f77b4"), ("DEP", "Départs", "#ff7f0e")]):
    g = air[air.sens == sens].groupby("bin")
    for i, (var, lab) in enumerate([("alt_ft", "Altitude (ft)"), ("gs_kt", "Vitesse sol (kt)")]):
        q = g[var].quantile([.1, .5, .9]).unstack()
        ax[i, j].fill_between(q.index, q[0.1], q[0.9], color=col, alpha=.25)
        ax[i, j].plot(q.index, q[0.5], color=col, lw=2)
        ax[i, j].set_ylabel(lab); ax[i, j].grid(alpha=.3)
    ax[0, j].set_title(titre); ax[1, j].set_xlabel("Distance à l'aéroport (NM)")
    if sens == "ARR":
        ax[0, j].invert_xaxis()
plt.tight_layout(); plt.show()

phases = (traj.groupby(["sens", "phase"]).agg(points=("alt_ft", "size"), alt_med=("alt_ft", "median"),
                                               vitesse_med_kt=("gs_kt", "median"),
                                               vitesse_max_kt=("gs_kt", "max")).round(0))
phases

## 7. Carte 3D animée
- **Vue d'ensemble** : toutes les trajectoires de la journée, colorées par altitude.
- **Animation** : un curseur temporel fait évoluer les avions (étiquette = vol, compagnie, type, altitude, vitesse) avec leur traîne ; ils disparaissent dès qu'ils sortent de la zone.

*L'axe vertical est exagéré pour que les profils soient lisibles (le relief réel est quasi plat à cette échelle).*

In [ ]:
import plotly.graph_objects as go

R_KM = ZONE_NM * KM_PER_NM
th = np.linspace(0, 2 * np.pi, 181)
U = np.array([np.sin(np.radians(RWY_HDG)), np.cos(np.radians(RWY_HDG))])
rwy = np.array([-RWY_HALF_KM * U, RWY_HALF_KM * U])
Z_SCALE = 0.3048                                    # ft -> m, affiché en km (z = alt_ft × 0.3048 / 1000)

def zkm(alt_ft):
    return np.asarray(alt_ft) * 0.3048 / 1000

def static_traces():
    return [
        go.Scatter3d(x=R_KM * np.cos(th), y=R_KM * np.sin(th), z=np.zeros_like(th), mode="lines",
                     line=dict(color="#888", width=3, dash="dash"), name="Zone 25 NM", hoverinfo="skip"),
        go.Scatter3d(x=rwy[:, 0], y=rwy[:, 1], z=[0, 0], mode="lines", line=dict(color="black", width=8),
                     name="Piste 12/30", hoverinfo="skip"),
    ]

def scene(title):
    return dict(
        title=title, height=720, margin=dict(l=0, r=0, t=50, b=0), legend=dict(orientation="h"),
        scene=dict(xaxis=dict(title="Est (km)", range=[-R_KM, R_KM]), yaxis=dict(title="Nord (km)", range=[-R_KM, R_KM]),
                   zaxis=dict(title="Altitude (km)", range=[0, 5]),
                   aspectmode="manual", aspectratio=dict(x=1, y=1, z=0.45),
                   camera=dict(eye=dict(x=1.1, y=-1.5, z=0.9))))

# ---- 7a. Vue d'ensemble : toutes les trajectoires, couleur = altitude
xs, ys, zs, cs = [], [], [], []
for fid, g in traj.groupby("flight_id"):
    xs += list(g.x_km) + [np.nan]; ys += list(g.y_km) + [np.nan]
    zs += list(zkm(g.alt_ft)) + [np.nan]; cs += list(g.alt_ft) + [np.nan]
fig = go.Figure(static_traces() + [go.Scatter3d(
    x=xs, y=ys, z=zs, mode="lines", name="Trajectoires",
    line=dict(color=cs, colorscale="Turbo", width=3, cmin=0, cmax=14000,
              colorbar=dict(title="Altitude (ft)", len=0.6)), hoverinfo="skip")])
fig.update_layout(**scene("Toutes les trajectoires – LFOB, 12/10/2026"))
fig.show()

In [ ]:
# ---- 7b. Animation avec curseur temporel
# Réglages : fenêtre de temps à animer (par défaut : l'heure la plus chargée)
START_H = int(par_heure.Total.idxmax())   # ex. 14 pour 14h ; mettre un entier pour imposer l'heure
DURATION_MIN = 60                          # durée de la fenêtre animée
STEP_S = 20                                # pas entre deux images (secondes)
TRAIL_S = 150                              # longueur de la traîne derrière chaque avion

groups = {fid: g for fid, g in traj.groupby("flight_id")}
T0 = START_H * 3600
Ts = np.arange(T0, T0 + DURATION_MIN * 60 + 1, STEP_S)

def frame_data(T):
    px, py, pz, txt, hov, col = [], [], [], [], [], []
    tx, ty, tz = [], [], []
    for fid, g in groups.items():
        t = g.t_local_s.values
        if not (t[0] <= T <= t[-1]):
            continue
        x = np.interp(T, t, g.x_km); y = np.interp(T, t, g.y_km)
        alt = np.interp(T, t, g.alt_ft); gs = np.interp(T, t, g.gs_kt)
        i = min(np.searchsorted(t, T), len(t) - 1)
        r = g.iloc[i]
        px.append(x); py.append(y); pz.append(zkm(alt)); col.append("#1f77b4" if r.sens == "ARR" else "#ff7f0e")
        txt.append(r.vol)
        hov.append(f"<b>{r.vol}</b> – {r.compagnie}<br>{r.type_avion} ({r.immat})<br>"
                   f"{'Arrivée de' if r.sens == 'ARR' else 'Départ vers'} {r.aeroport_lie_ville}<br>"
                   f"Alt {alt:,.0f} ft · {gs:.0f} kt<br>Phase : {r.phase}")
        m = (t >= T - TRAIL_S) & (t <= T)
        tx += list(g.x_km.values[m]) + [x, np.nan]; ty += list(g.y_km.values[m]) + [y, np.nan]
        tz += list(zkm(g.alt_ft.values[m])) + [zkm(alt), np.nan]
    trail = go.Scatter3d(x=tx, y=ty, z=tz, mode="lines", line=dict(color="rgba(80,80,80,.7)", width=4),
                         name="Traîne", hoverinfo="skip", showlegend=False)
    planes = go.Scatter3d(x=px, y=py, z=pz, mode="markers+text", text=txt, textposition="top center",
                          marker=dict(size=6, color=col, line=dict(color="black", width=1)),
                          hovertext=hov, hoverinfo="text", name="Avions", showlegend=False)
    return [trail, planes]

label = lambda T: f"{int(T // 3600):02d}:{int(T % 3600 // 60):02d}:{int(T % 60):02d}"
frames = [go.Frame(data=frame_data(T), traces=[2, 3], name=label(T)) for T in Ts]
fig = go.Figure(static_traces() + frame_data(Ts[0]), frames=frames)
fig.update_layout(**scene(f"Trafic animé – de {label(Ts[0])} à {label(Ts[-1])} (heure locale)"))
fig.update_layout(
    updatemenus=[dict(type="buttons", x=0.05, y=0.02, buttons=[
        dict(label="▶ Lecture", method="animate",
             args=[None, dict(frame=dict(duration=120, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
        dict(label="⏸ Pause", method="animate",
             args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
    sliders=[dict(x=0.15, len=0.8, currentvalue=dict(prefix="Heure locale : "),
                  steps=[dict(label=label(T), method="animate",
                              args=[[label(T)], dict(mode="immediate", frame=dict(duration=0, redraw=True),
                                                     transition=dict(duration=0))]) for T in Ts])])
fig.show()
print(f"{len(Ts)} images – fenêtre {label(Ts[0])} → {label(Ts[-1])}. Changez START_H / DURATION_MIN pour une autre période.")

## 8. Export pour l'application Streamlit
On écrit des tables propres, légères, et prêtes à lire par l'application :
- `app_trajectoires.csv` – points de trajectoire avec toutes les infos vol ;
- `app_vols.csv` – une ligne par vol ;
- `app_trafic_par_heure.csv` – mouvements par heure ;
- `app_profils_phase.csv` – statistiques par phase de vol.

In [ ]:
out = {
    "app_trajectoires.csv": traj.drop(columns=["vol_y"], errors="ignore"),
    "app_vols.csv": vols.drop(columns=["heure"]),
    "app_trafic_par_heure.csv": par_heure.reset_index().rename(columns={"heure": "heure_locale"}),
    "app_profils_phase.csv": phases.reset_index(),
}
for name, df in out.items():
    df.to_csv(name, index=False)
    print(f"{name:28s} {len(df):>6} lignes")

if IN_COLAB:                                  # téléchargement automatique depuis Colab
    for name in out:
        files.download(name)

## Pour aller plus loin avec de vraies données
Pour remplacer le jeu fictif par de vraies trajectoires (Flightradar24 / ADS-B), il suffit de fournir deux fichiers avec **les mêmes colonnes** :

| Fichier | Colonnes indispensables |
|---|---|
| trajectoires | `flight_id, lat, lon, alt_ft, gs_kt, t_local_s, heading_deg, phase, on_ground, dist_arp_nm` |
| vols | `flight_id, vol, sens (ARR/DEP), compagnie, type_avion, immat, aeroport_lie_ville, corridor, heure_evenement_retenue` |

Le reste du notebook (contrôles, analyses, carte 3D, export) fonctionne sans modification.